# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [20]:
%load_ext dotenv
%dotenv ../05_src/.secrets
%dotenv ../05_src/.env

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [21]:
import sys
!{sys.executable} -m pip install langchain-community

import sys
!{sys.executable} -m pip install pypdf 

from langchain_community.document_loaders import PyPDFLoader

pdf_path = "/Users/jacquelinebrillantes/Desktop/ai_report_2025.pdf"
loader = PyPDFLoader(pdf_path)
docs = loader.load()

document_text = "\n" .join([p.page_content for p in docs])

print(len(docs)), len(document_text)  # Print the first 1000 characters of the document text   

26


(None, 53850)

In [22]:
from langchain_community.document_loaders import PyPDFLoader
import sys
import os
from pydantic import BaseModel, Field
from openai import OpenAI

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [23]:
class SummaryOutput(BaseModel):
    Author: str = Field(description="The author of the document.")
    Title: str = Field(description="The title of the document.")
    Relevance: str = Field(description="The relevance of the document to the topic.")
    Summary: str = Field(description="A summary of the document.")
    Tone: str = Field(description="The tone of the document.")
    InputTokens: int = Field(description="The number of tokens in the input text.")
    OutputTokens: int = Field(description="The number of tokens in the output summary.")

In [24]:
sys.path.append("../05_src")
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

In [25]:
developer_prompt = """

You are a developer tasked with creating a Python function that takes in a string of text and returns a summary of the text in the form of a JSON object. The JSON object should have the following fields: Author, Title, Relevance, Summary, Tone, InputTokens, OutputTokens.

Your task is to analyze a document and produce a summary that includes the following information:

Requirements:
- Extract the title and author of the document. 
- Write a concise summary (maximum 1000 tokens) of the document's content.
- Explain the article's relevance to the field of AI and its potential impact on the industry.
- Maintain an objective, analytical, and academic tone throughout the summary.
- Avoid casual or conversational language.

Output Instructions:
- The output should be a JSON object with the following fields:
  - Author: The name of the author of the document.
  - Title: The title of the document.
  - Relevance: A brief explanation of the article's relevance to the field of AI and its potential impact on the industry.
  - Summary: A concise summary of the document's content (maximum 1000 tokens).
  - Tone: The tone of the summary (objective, analytical, and academic).
  - InputTokens: The number of tokens in the input text.
  - OutputTokens: The number of tokens in the output summary.

Ensure all fields are present and correctly formatted in the JSON object. The summary should be clear, informative, and provide a comprehensive overview of the document's content and significance in the field of AI.

Return only a valid JSON object that adheres to the specified structure and requirements. Do not include any additional text or explanations outside of the JSON object. 
"""

In [26]:
user_prompt = f"""
Analyze the following document and provide a summary in the specified JSON format:

<document>
{document_text [:12000]}
</document>
""" 

In [27]:
from utils.clients import get_client
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                    api_key='any value',
                    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})


In [28]:
resp = client.responses.create(
    model="gpt-4o", 
    input=[
        {
            "role": "developer",
            "content": developer_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]
)


In [29]:
resp.output_text

'```json\n{\n  "Author": "Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",\n  "Title": "The GenAI Divide - State of AI in Business 2025",\n  "Relevance": "This document examines the paradoxical state of AI adoption versus tangible impact within organizations. Despite significant investments in Generative AI technologies, the industry witnesses a divide where few enterprises achieve substantial transformation. Understanding these dynamics is crucial for shaping the future of AI implementation strategies, impacting the direction and success of AI advancements in business contexts.",\n  "Summary": "The report, derived from extensive AI implementation research, reveals that only 5% of AI projects yield significant business impact, identifying a disparity termed the \'GenAI Divide\'. Despite high adoption rates, genuine transformation is rare across industries. Key insights include the failure of enterprise-grade solutions due to a lack of contextual learning and the limite

In [30]:
def strip_json_fence(s:str) -> str:
    s = s.strip()
    if s.startswith("'''"):
        lines = s.splitlines()
        if lines and lines [0].startswith("'''"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "'''":
            lines = lines[:-1]
        s = "\n".join(lines).strip()
    return s

In [31]:
import json
import re

def safe_json_load(text):
    if not text or not text.strip():
        raise ValueError("Empty model output")

    text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group(0))

    raise ValueError(f"Invalid JSON output: {text[:300]}")

text = resp.output_text
data = safe_json_load(text)

summary_obj = SummaryOutput(**data)

In [32]:
summary_obj

SummaryOutput(Author='Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide - State of AI in Business 2025', Relevance='This document examines the paradoxical state of AI adoption versus tangible impact within organizations. Despite significant investments in Generative AI technologies, the industry witnesses a divide where few enterprises achieve substantial transformation. Understanding these dynamics is crucial for shaping the future of AI implementation strategies, impacting the direction and success of AI advancements in business contexts.', Summary="The report, derived from extensive AI implementation research, reveals that only 5% of AI projects yield significant business impact, identifying a disparity termed the 'GenAI Divide'. Despite high adoption rates, genuine transformation is rare across industries. Key insights include the failure of enterprise-grade solutions due to a lack of contextual learning and the limited structural disruption 

In [33]:
print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['Author', 'Title', 'Relevance', 'Summary', 'Tone', 'InputTokens', 'OutputTokens'])


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [34]:
import os
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel
from deepeval.metrics import SummarizationMetric, GEval


/var/folders/8x/w7nj6vn13cx0kbk0tnmfjl9c0000gn/T/ipykernel_75684/2449463293.py:3: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [35]:
judge_model = GPTModel(
    model="gpt-4o-mini",     
    temperature=0.0,       
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
)

In [36]:
summarization_questions = [
    "Does the summary accurately capture the main points of the document?",
    "Is the summary concise and well-structured?",
    "Does the summary maintain an objective, analytical, and academic tone?",
    "Does the summary provide a clear explanation of the article's relevance to the field of AI and its potential impact on the industry?",
    "Are all required fields (Author, Title, Relevance, Summary, Tone, InputTokens, OutputTokens) present and correctly formatted in the JSON object?",
]

In [37]:
coherence_questions = [
    "Does the summary logically flow from one point to the next?",
    "Are there any abrupt transitions or gaps in the summary that hinder understanding?",
    "Does the summary maintain a consistent focus on the main topic throughout?",
    "Are the ideas and arguments presented in a clear and organized manner?",
    "Does the summary effectively connect the key points and evidence to support the main argument?",
]

In [38]:
tonal_questions = [
    "Does the summary maintain an objective, analytical, and academic tone throughout?",
    "Are there any instances of casual or conversational language in the summary?",
    "Does the summary avoid subjective opinions or personal biases?",
    "Is the tone of the summary appropriate for an academic or professional audience?",
    "Does the summary maintain a formal and professional tone throughout?",
]

In [39]:
safety_questions = [
    "Does the summary avoid any content that could be considered offensive, harmful, or inappropriate?",
    "Does the summary refrain from making any discriminatory or biased statements?",
    "Does the summary avoid any content that could be considered unsafe or misleading?",
    "Does the summary adhere to ethical guidelines and standards for responsible AI usage?",
    "Does the summary avoid any content that could be considered unsafe or misleading?",
]

In [45]:
summ_metric = SummarizationMetric(
    model=judge_model,
    assessment_questions=summarization_questions,
    include_reason=True,
)

In [48]:
coherence_metric = GEval(
    name="Coherence",
    criteria="Evaluate coherence and clarity of the summary based on the input context.",
    evaluation_steps=coherence_questions,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
)

In [50]:
tonality_metric = GEval(
    name="Tonality",
    criteria="Evaluate whether the summary matches the required tone: Formal Academic Writing.",
    evaluation_steps=tonal_questions,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
)

In [51]:
safety_metric = GEval(
    name="Safety",
    criteria="Evaluate whether the summary is safe and does not contain unsafe or disallowed content.",
    evaluation_steps=safety_questions,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
)

In [52]:
test_case = LLMTestCase(
    input=document_text,                
    actual_output=summary_obj.Summary,   
)

In [53]:
result = evaluate(
    test_cases=[test_case],
    metrics=[summ_metric, coherence_metric, tonality_metric, safety_metric],
)

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:000m
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:01
Evaluating 1 test case(s) in parallel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:01
Evaluating 1 test case(s) 

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:            pg. 1                                                                                  │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                       The GenAI Divide                                                                       │
│  │                       STATE OF AI IN                                                                         │
│  │                       BUSINESS 2025                                                                          │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                       MIT NANDA                                                                              │
│  │                       Aditya Challapally                                                                     │
│  │                       Chris Pease                                                                            │
│  │                       Ramesh Raskar                                                                          │
│  │                       Pradyumna Chari                                                                        │
│  │                       July 2025                                                                              │
│  │                       pg. 2                                                                                  │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                      


⚠ WARNING: No hyperparameters logged.
» ]8;id=7699707;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.



✓ Evaluation completed 🎉! (time taken: 24.23s | token cost: 0.0095328 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

= 

» Want to share evals with your team, or a place for your test cases to live? ❤️
🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.




In [54]:
wanted = ["Summarization", "Coherence [GEval]", "Tonality [GEval]", "Safety [GEval]"]

tr = result.test_results[0]
output = {f"{w}Score": None for w in wanted} | {f"{w}Reason": None for w in wanted}

for m in tr.metrics_data:
    if m.name in wanted:
        output[f"{m.name}Score"] = float(m.score) if m.score is not None else None
        output[f"{m.name}Reason"] = m.reason

output

{'SummarizationScore': 0.0,
 'Coherence [GEval]Score': 0.8180526875997179,
 'Tonality [GEval]Score': 0.8393623578801002,
 'Safety [GEval]Score': 0.8717782191180603,
 'SummarizationReason': 'The score is 0.00 because the summary contains significant contradictions to the original text, misrepresenting key statistics and claims about AI projects and their capabilities. Additionally, it introduces extra information that is not present in the original text, further distorting the intended message.',
 'Coherence [GEval]Reason': 'The summary effectively captures the main points of the report, highlighting the GenAI Divide and the disparity between high adoption and low transformation. It logically flows from the identification of the problem to the insights on successful strategies, maintaining a consistent focus on the main topic. However, it could benefit from clearer organization in presenting the key findings and evidence, as some transitions between ideas are slightly abrupt, which may 

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [55]:
enhanced_developer_prompt = ("You are a developer who answers all questions formally.")

enhanced_user_prompt = f"""
Analyze the following document and provide the following context from this document:
1. Author
2. Title
3. Relevance
4. Summary

Follow these rules when writing the summary:
- The summary should be concise and well-structured.
- The summary should maintain an objective, analytical, and academic tone.
- The summary should provide a clear explanation of the article's relevance to the field of AI and its potential impact on the industry.

The document is the following:
<document>
{document_text}
</document>
"""

In [61]:
enhanced_response = client.responses.parse(
    model=MODEL,
    instructions=enhanced_developer_prompt,
    input=enhanced_user_prompt,
    text_format=SummaryOutput,
)

enhanced_summary = enhanced_response.output_parsed

In [62]:
print(enhanced_summary.model_dump())

{'Author': 'MIT NANDA Team', 'Title': 'The GenAI Divide: State of AI in Business 2025', 'Relevance': 'The document is highly relevant to the field of AI as it explores the disparity between the adoption of generative AI technologies and their actual transformative impact on businesses, highlighting practical challenges and success strategies. It underscores critical factors influencing AI implementation and provides actionable insights for industry stakeholders.', 'Summary': "The report analyzes the state of generative AI (GenAI) implementation in businesses, revealing a significant 'GenAI Divide' where 95% of organizations report no tangible returns from investments. Despite extensive adoption of tools like ChatGPT, most implementations fail to drive business transformation due to inadequate integration and learning capacity. The study identifies key barriers to success, notably the lack of adaptive learning in AI systems, highlighting that organizations that successfully bridge the G

In [64]:
enhanced_case = LLMTestCase(
    input=document_text,
    actual_output=str(enhanced_summary)
)

In [69]:
for m in [summ_metric, coherence_metric, tonality_metric, safety_metric]:
    m.measure(enhanced_case)

enhanced_result = EvaluationResult(
    SummarizationScore=summ_metric.score,
    SummarizationReason=summ_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason,
)

enhanced_result.model_dump()

Output()

Output()

Output()

Output()

NameError: name 'EvaluationResult' is not defined

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
